In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=2)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="CalculateMoreVariables", dataName="TimeSeries",
                                dtype='float32')

In [ ]:
####################################
#DATA LOADING FUNCTIONS

In [ ]:
def LoadOrRun(function, fileName="timeSeriesData.pkl"):
    """
    Loads data from DataManager.outputDataDirectory if it exists.
    Otherwise, runs the provided function and saves the output.
    """
    # Construct the full file path
    filepath = os.path.join(DataManager.outputDataDirectory, fileName)
    
    # Check if the file already exists
    if os.path.exists(filepath):
        print(f"Loading cached data from: {filepath}")
        with open(filepath, 'rb') as file:
            data = pickle.load(file)
        return data
    else:
        print("Data not found. Running calculation...")
        # Run the target function
        data = function()
        
        # Ensure the output directory exists
        os.makedirs(DataManager.outputDataDirectory, exist_ok=True)
        
        # Save the result for future use
        print(f"Saving computed data to: {filepath}")
        with open(filepath, 'wb') as file:
            pickle.dump(data, file)
            
        return data

In [ ]:
def safeMean(ModelData, DataManager, timeString, varname):
    
    if varname in ['winterp_u', 'winterp_d', 'buoyancy_u', 'buoyancy_d']:
        result = CallVariable(ModelData, DataManager, timeString, varname.replace('_u','').replace('_d',''))
        result = result[0:np.abs(ModelData.zh-2).argmin()+1]
        threshold = 0.1 if varname.startswith('winterp') else 0.005
        mask = result > threshold if varname.endswith('_u') else result < -threshold
        
        return np.mean(result[mask]) if mask.any() else np.nan
    
    elif varname in ['winterp', 'buoyancy']:
        result = CallVariable(ModelData, DataManager, timeString, varname)
        return np.mean(result[0:np.abs(ModelData.zh-1).argmin()+1])
    
    else:
        result = CallVariable(ModelData, DataManager, timeString, varname)
        return np.mean(result[0] if result.ndim > 2 else result)
        
def CalculateTimeSeries():
    
    varnames = GetVarNames()
    timeSeriesDictionary = {varname: [] for varname in varnames}
    
    for timeString in tqdm(ModelData.timeStrings):
        varDataDictionary = {varname: safeMean(ModelData, DataManager, timeString, varname) for varname in varnames}
        for varname, value in varDataDictionary.items(): timeSeriesDictionary[varname].append(value)

    return timeSeriesDictionary

def GetVarNames():
    varnames=["th","theta_v","qr","qv"]
    varnames+=["winterp","winterp_u","winterp_d"]
    varnames+=["buoyancy","buoyancy_u","buoyancy_d"]
    varnames+=["theta_e"]
    varnames+=['qvflux','thflux']
    return varnames

unitsDictionary = {f"th": r"$(K)$",f"theta_v": r"$(K)$",f"theta_e": r"$(K)$",
                   f"qv": r"$(g\ kg^{-1})$",f"qr": r"$(g\ kg^{-1})$",
                   f"winterp": r"$(m\ s^{-1})$", f"buoyancy": r"$(m\ s^{-1}/s)$",
                   f"thflux": r"$K\ m\ s^{-1}$",f"qvflux": r"$g\ kg^{-1}\ m\ s^{-1}$"}

In [ ]:
####################################
#DATA LOADING

In [ ]:
fileName=os.path.join(DataManager.outputDataDirectory,"timeSeriesData.pkl")
timeSeriesDictionary = LoadOrRun(function=CalculateTimeSeries,fileName=fileName)

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def MakePlot(timeSeriesDictionary):
    varnames = GetVarNames()
    plotVarnames = list(dict.fromkeys([v.replace('_u','').replace('_d','') for v in varnames]))  # unique bases, order preserved
    time = np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    
    nPlots = len(plotVarnames)
    nCols = 3
    nRows = int(np.ceil(nPlots/nCols))
    fig, axes = plt.subplots(nRows, nCols, figsize=(15, 4*nRows), sharex=True, gridspec_kw={'wspace':0.15,'hspace':0.15})
    axes = axes.flatten()
    
    for i, varname in enumerate(varnames):
        data = timeSeriesDictionary[varname]
        color = 'blue' if varname.endswith('_u') else 'green' if varname.endswith('_d') else 'black'
        multiplier = 1000 if 'q' in varname else 1
        base = varname.replace('_u','').replace('_d','')
        units = unitsDictionary[base]
        ax = axes[plotVarnames.index(base)]
        ax.plot(time, multiplier*np.array(data), color=color)
        if (plotVarnames.index(base) // nCols) == nRows-1: ax.set_xlabel("time (hrs)")
        titleString = " (0-2 km)" if base in ['winterp','buoyancy'] else ""
        ax.set_title(f"{base}{titleString} {units}")


        if base in ['qr','winterp','buoyancy']:
            ax.axhline(0,color='gray',zorder=-10)
            apply_scientific_notation([ax],dimension='y')
        if base in ['qr']:
            ax.set_ylim(bottom=0)
    for ax in axes[len(plotVarnames):]: ax.set_visible(False)
    return fig

In [ ]:
def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
####################################
#PLOTTING

In [ ]:
fig = MakePlot(timeSeriesDictionary)
SaveFigure(fig,plotType="Variable_Calculation/CalculateMoreVariables/TimeSeries",fileName="TimeSeries")